## Caracteristicas de navegación en ventas online ##

### Definición del problema y objetivos ###
**Grupo 2**: Renzo Vilchez\
**Asignatura**: MCDI501 - Estadística computacional para la toma de decisiones.\
**Repositorio**: https://github.com/RenzoVilchez02/proyectoGrupo2

### Ejecución ###



In [5]:
# Comprobación de requirements.txt.
from pathlib import Path as _Path
from importlib import metadata as _metadata
import sys as _sys

_inicio = _Path.cwd().resolve()
_raiz_entorno = next(
    (ruta for ruta in (_inicio, *_inicio.parents)
     if (ruta / "requirements.txt").is_file() and (ruta / "F1").is_dir() ),
    None,
)
if _raiz_entorno is None:
    raise RuntimeError("Abra el notebook dentro de la carpeta raíz del proyecto grupo 2.")
_fijadas = {}
for _linea in (_raiz_entorno / "requirements.txt").read_text(encoding="utf-8").splitlines():
    _linea = _linea.split("#", 1)[0].strip()
    if "==" in _linea:
        _paquete, _version = _linea.split("==", 1)
        _fijadas[_paquete.strip().lower()] = _version.strip()
_distintas = {}
for _paquete, _esperada in _fijadas.items():
    try:
        _actual = _metadata.version(_paquete)
    except _metadata.PackageNotFoundError:
        _actual = "no instalado"
    if _actual != _esperada:
        _distintas[_paquete] = f"{_actual} (requiere {_esperada})"
if _distintas:
    raise RuntimeError(
        "Kernel equivocado: las versiones no coinciden con requirements.txt. "
        f"Diferencias: {_distintas}. Intérprete actual: {_sys.executable}. "
        "En VS Code, arriba a la derecha: Select Kernel > Jupyter Kernels > "
        "Grupo 2 requirements (Python 3.14.7), o elija el Python de .venv. "
        "Seleccione ese entorno y vuelva a ejecutar desde la primera celda."
    )
print("Entorno correcto: versiones fijadas en requirements.txt verificadas.")

import sys
import json
import platform
import hashlib
import subprocess
import shutil
import importlib.util
from importlib import metadata
from pathlib import Path
from datetime import date

try:
    import pandas as pd
except ImportError as error:
    raise ImportError("Falta pandas en el kernel seleccionado. Instálelo en el entorno del proyecto.") from error

print("F1 — fecha de ejecución:", date.today().isoformat())

Entorno correcto: versiones fijadas en requirements.txt verificadas.
F1 — fecha de ejecución: 2026-10-07


In [6]:
def verificar_entorno():
    """Devuelve versiones reales y condiciones del intérprete, sin instalar paquetes."""
    versiones = {}
    for paquete in ("pandas", "numpy", "matplotlib", "ipykernel", "jupyterlab"):
        try:
            versiones[paquete] = metadata.version(paquete)
        except metadata.PackageNotFoundError:
            versiones[paquete] = None
    return {
        "python": platform.python_version(), "interprete": Path(sys.executable).name,
        "sistema": platform.system(),
        "entorno_virtual_detectado": sys.prefix != sys.base_prefix or (Path(sys.prefix) / "conda-meta").is_dir(),
        "versiones": versiones,
        "confirmacion_kernel_equipo": "Entorno y ejecución cruzada revisados por el equipo",
    }

ENTORNO = verificar_entorno()
print(json.dumps(ENTORNO, indent=2, ensure_ascii=False))

{
  "python": "3.14.7",
  "interprete": "python.exe",
  "sistema": "Windows",
  "entorno_virtual_detectado": true,
  "versiones": {
    "pandas": "3.0.5",
    "numpy": "2.5.3",
    "matplotlib": "3.11.2",
    "ipykernel": "7.3.0",
    "jupyterlab": "4.6.3"
  },
  "confirmacion_kernel_equipo": "Entorno y ejecución cruzada revisados por el equipo"
}


In [7]:
def encontrar_raiz(inicio):
    """Localiza una raíz con carpetas F1 y data; falla si no existe."""
    inicio = Path(inicio).resolve()
    for candidato in (inicio, *inicio.parents):
        if all((candidato / nombre).is_dir() for nombre in ("F1", "data")):
            return candidato
    raise FileNotFoundError("Abra el notebook dentro del repositorio que contiene F1/ y data/.")

RAIZ = encontrar_raiz(Path.cwd())
DIR_DOCS = RAIZ / "docs" / "f1"
DIR_SRC = RAIZ / "src"
for relativo in ("data/raw", "data/processed", "docs/f1", "src", "F1", "outputs"):
    (RAIZ / relativo).mkdir(parents=True, exist_ok=True)
ARCHIVO = RAIZ / "data" / "raw" / "online_shoppers_intention.csv"
print("Raíz:", RAIZ)
for relativo in ("data/raw", "data/processed", "docs/f1", "src", "F1", "outputs"):
    print(relativo + "/")

ESTADO_ARCHIVOS = {}
for nombre in ("README.md", ".gitignore", "requirements.txt"):
    ruta = RAIZ / nombre
    estado = "presente con contenido" if ruta.exists() and ruta.read_text(encoding="utf-8").strip() else "ausente"
    ESTADO_ARCHIVOS[nombre] = estado
print(ESTADO_ARCHIVOS)

Raíz: C:\Users\vilch\proyectoGrupo2
data/raw/
data/processed/
docs/f1/
src/
F1/
outputs/
{'README.md': 'presente con contenido', '.gitignore': 'ausente', 'requirements.txt': 'presente con contenido'}


### 1. Definición del problema ###
En este proyecto se analizará un conjunto de datos sobre sesiones realizadas en un sitio de comercio electrónico. El dataset utilizado es Online Shoppers Purchasing Intention Dataset, disponible en el repositorio de la UCI (University of California, Irvine), y contiene información de 12.330 sesiones de navegación.

El objetivo del proyecto es analizar si existe alguna relación entre el comportamiento de navegación de los usuarios y la realización de una compra. Para esto, se estudiarán principalmente las variables relacionadas con las páginas administrativas, informativas y de productos, sus tiempos de navegación, la tasa de rebote y la tasa de salida.

In [ ]:
PROYECTO = {'titulo': 'Caracteristicas de navegación en ventas online',
 'grupo': 'Grupo 2',
 'asignatura': 'MCDI501',
 'integrantes': ['Renzo Vilchez'],
 'repositorio': 'https://github.com/RenzoVilchez02/proyectoGrupo2',
 'problematica': 'En un sitio de comercio electrónico, no todas las sesiones de navegación terminan en una compra.',
 'objetivo_general': 'Analizar si existe una relación estadísticamente significativa entre las características de navegación de una sesión y la probabilidad de que esta termine en una compra.',
 'objetivos_especificos': ['Explorar y describir las principales características del conjunto de datos y de las variables relacionadas con el comportamiento de navegación.',
                           'Comparar las características de navegación entre las sesiones que terminan en una compra y aquellas que no terminan en una compra.',
                           'Analizar la relación entre el número de páginas visitadas y la realización de una compra.',
                           'Analizar la relación entre el tiempo de navegación en los diferentes tipos de páginas y la realización de una compra.',
                           'Evaluar la relación entre las tasas de rebote y salida y la realización de una compra.',
                           'Determinar mediante pruebas estadísticas si las relaciones o diferencias observadas son estadísticamente significativas.',
                           'Identificar cuáles de las características de navegación analizadas presentan una mayor relación con la probabilidad de que una sesión termine en una compra.'],
 'preguntas': ['¿Existe una diferencia significativa en el número de páginas de productos visitadas entre las sesiones que terminan en compra y las que no?',
               '¿Existe una diferencia significativa en el tiempo dedicado a páginas de productos entre las sesiones que terminan en compra y las que no?',
               '¿Existe una diferencia significativa en la tasa de rebote entre las sesiones que terminan en compra y las que no?'],
 'criterios_exito': ['F1 puede ejecutarse en el entorno documentado desde un kernel '
                     'reiniciado.',
                     'Las decisiones de preparación se justifican con conteos y '
                     'comparaciones.',
                     'Los integrantes mantienen contribuciones propias trazables en Git.',
                     'Los indicadores usan el período completo y denominadores documentados.'],
 'alcance_f1': {'incluye': ['Preparación y carga de datos',
                            'Análisis exploratorio de datos',
                            'Estimación de parametros',
                            'Pruebas de hipotesis'],
                'excluye': ['Predicción de compras futuras']},
 'limitaciones': ['La mayoría de las sesiones no terminan en una compra, mientras que aproximadamente el 15% termina en una compra. Esto puede dificultar la comparación entre ambos grupos y afectar algunos análisis estadísticos.',
                  'Existen variables que están relacionadas entre sí. Por ejemplo, el número de páginas de productos visitadas (ProductRelated) está relacionado con el tiempo dedicado a esas páginas (ProductRelated_Duration). Lo mismo ocurre con BounceRates y ExitRates. Esto puede generar problemas de multicolinealidad al utilizar varias variables en un mismo modelo.',
                  'El dataset permite estudiar relaciones y asociaciones entre las variables, pero no permite afirmar que una determinada característica de navegación sea la causa de que un usuario realice una compra.',
                  'Los datos se centran principalmente en las características de las sesiones de navegación. No se dispone de información más detallada sobre los usuarios, como edad, ingresos, preferencias personales o motivos de compra.',
                  'Los datos corresponden a sesiones registradas durante un año. Por lo tanto, los resultados podrían estar condicionados por las características de ese periodo y no necesariamente representar el comportamiento de los usuarios en otros años.',
                  'Aunque el dataset no presenta valores faltantes, esto no garantiza que todos los datos sean suficientes para explicar el comportamiento de los usuarios. Puede existir información relevante que simplemente no fue registrada.'],
'supuestos': ['Se asume que cada registro corresponde a una sesión independiente y que el comportamiento de una sesión no afecta directamente al de otra.',
              'Se asume que las variables del dataset fueron registradas de forma correcta y que los valores representan adecuadamente las acciones realizadas durante cada sesión.',
              'Se asume que cuando Revenue es True, la sesión terminó efectivamente en una compra, y cuando es False, no se produjo una compra.',
              'Se asume que variables como el número de páginas visitadas, el tiempo de navegación, la tasa de rebote y la tasa de salida pueden contener información útil para analizar la probabilidad de conversión.',
              'Se asume que las 12.330 sesiones disponibles contienen suficiente información para estudiar la relación entre el comportamiento de navegación y la realización de una compra, aunque los resultados estarán limitados al contexto del dataset.']}

def presentar_proyecto(config):
    """Valida la definición mínima y muestra las preguntas del proyecto."""
    requeridas = ("titulo", "grupo", "integrantes", "objetivo_general", "preguntas")
    faltantes = [clave for clave in requeridas if not config.get(clave)]
    if faltantes:
        raise ValueError(f"Definición incompleta: {faltantes}")
    print(config["titulo"])
    print(config["objetivo_general"])
    for numero, pregunta in enumerate(config["preguntas"], 1):
        print(f"{numero}. {pregunta}")
    return True

presentar_proyecto(PROYECTO)

Caracteristicas de navegación en ventas online
Analizar si existe una relación estadísticamente significativa entre las características de navegación de una sesión y la probabilidad de que esta termine en una compra.
1. ¿Existe una relación significativa entre las características de navegación de una sesión y la probabilidad de que esta termine en una compra?


True

## Definición de los indicadores ##

| Pregunta | Numerador o medida | Denominador o alcance |
| --- | --- | --- |
| Numero de paginas | ProductRelated, Revenue | Sesiones con compra vs. sesiones sin compra |
| Tiempo dedicado | ProductRelated_Duration, Revenue | Sesiones con compra vs. sesiones sin compra |
| Tasa de rebote | BounceRates, Revenue | Sesiones con compra vs. sesiones sin compra |

Condiciones de reutilización y redistribución
El dataset proviene del portal UC Irvine Machine Learning Repository:
https://archive.ics.uci.edu/dataset/468/online+shoppers+purchasing+intention+dataset

Se utilizará con fines académicos para analizar la relacion entre caracteristicas de compras online. La descarga se realizó el 07/10/2026. Se mantendrá la atribución a UC Irvine Machine Learning Repository y el archivo se conserva en data/raw/ para permitir la reproducción del análisis.